In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/harshmodi0710/metal-health/combined_dataset.csv
/kaggle/input/datasets/harshmodi0710/mental-health-2/demo.csv


In [2]:
df = pd.read_csv("/kaggle/input/datasets/harshmodi0710/mental-health-2/demo.csv")
df.head()

,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety


In [3]:
# ==============================
# INSTALL
# ==============================
# !pip install sentence-transformers scikit-learn pandas -q

# ==============================
# IMPORTS
# ==============================
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# ==============================
# LOAD DATA
# ==============================
df = pd.read_csv("/kaggle/input/datasets/harshmodi0710/mental-health-2/demo.csv")

# ==============================
# BALANCE DATASET (SAFE)
# ==============================
samples_per_class = 1500

df = df.groupby('status').apply(
    lambda x: x.sample(min(len(x), samples_per_class), random_state=42)
).reset_index(drop=True)

print("📊 Balanced Data:")
print(df['status'].value_counts())

# ==============================
# PREPARE DATA
# ==============================
texts = df['statement'].astype(str)
labels = df['status']

# ==============================
# TF-IDF FEATURES
# ==============================
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(texts)

# ==============================
# SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42
)

# ==============================
# TRAIN ML MODEL (IMPROVED)
# ==============================
clf = LogisticRegression(max_iter=2000)
clf.fit(X_train, y_train)

# ==============================
# EVALUATE
# ==============================
y_pred = clf.predict(X_test)
print("\n🔥 TF-IDF Accuracy:", accuracy_score(y_test, y_pred))
print("=" * 60)

# ==============================
# LOAD SENTENCE TRANSFORMER
# ==============================
model = SentenceTransformer('all-MiniLM-L6-v2')

# ==============================
# STRONG REFERENCE SET
# ==============================
ref_texts = [

# depression
"I feel empty inside",
"I feel hopeless and tired",
"I feel like giving up",
"I feel lost in life",
"I feel nothing anymore",

# anxiety
"I feel nervous all the time",
"My mind is racing",
"I feel stressed and overwhelmed",
"I cannot relax",
"I am constantly worried",

# fear
"I am scared",
"I feel afraid",
"I am terrified",
"I feel unsafe",

# anger
"I am very angry",
"I feel frustrated",
"I am irritated",
"I feel rage",

# sadness
"I feel sad",
"I feel low",
"I feel unhappy",

# normal
"I went to college",
"I had lunch",
"I did my work",
"I followed my routine",
"I had a normal day",

# happy
"I feel happy",
"I am enjoying life",
"I feel great",
"I am excited",
"I feel good"
]

ref_labels = [
"depression","depression","depression","depression","depression",
"anxiety","anxiety","anxiety","anxiety","anxiety",
"fear","fear","fear","fear",
"anger","anger","anger","anger",
"sadness","sadness","sadness",
"normal","normal","normal","normal","normal",
"happy","happy","happy","happy","happy"
]

# Encode once
ref_emb = model.encode(ref_texts, convert_to_tensor=True)

# ==============================
# PREDICTION FUNCTIONS
# ==============================
def predict_tfidf(text):
    vec = tfidf.transform([text])
    return clf.predict(vec)[0]

def predict_sbert(text):
    emb = model.encode([text], convert_to_tensor=True)
    scores = util.cos_sim(emb, ref_emb)
    idx = torch.argmax(scores).item()
    return ref_labels[idx]

# ==============================
# FINAL HYBRID DECISION
# ==============================
def final_predict(text):
    tfidf_pred = predict_tfidf(text)
    sbert_pred = predict_sbert(text)

    # If TF-IDF says normal but SBERT detects something else → trust SBERT
    if tfidf_pred.lower() == "normal" and sbert_pred != "normal":
        return sbert_pred

    # If both same → return
    if tfidf_pred.lower() == sbert_pred:
        return tfidf_pred

    # Otherwise → prefer SBERT
    return sbert_pred

# ==============================
# TEST CASES
# ==============================
tests = [
    "I smile but cry inside",
    "I feel very happy today",
    "I am nervous all the time",
    "I am angry right now",
    "I feel scared",
    "I went to college and came back",
    "I feel empty and tired",
    "My mind never stops thinking",
    "I am okay but something feels wrong"
]

print("\n🧪 FINAL RESULTS")
print("=" * 60)

for t in tests:
    print(f"Text: {t}")
    print(f"TF-IDF: {predict_tfidf(t)}")
    print(f"Sentence: {predict_sbert(t)}")
    print(f"Final: {final_predict(t)}")
    print("-" * 60)

# ==============================
# CHAT MODE
# ==============================
print("\n💬 CHAT MODE (type 'exit' to stop)\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    print("👉 Final Prediction:", final_predict(user_input))
    print("-" * 50)

/tmp/ipykernel_55/1770718990.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('status').apply(


📊 Balanced Data:
status
Anxiety                 1500
Bipolar                 1500
Depression              1500
Normal                  1500
Stress                  1500
Suicidal                1500
Personality disorder    1201
Name: count, dtype: int64

🔥 TF-IDF Accuracy: 0.7163155316021558


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

You:  i was facing some serious lung pain


👉 Final Prediction: Normal
--------------------------------------------------


You:  i was facing some serious lung pain because i am emotional


👉 Final Prediction: anxiety
--------------------------------------------------


You:  exit


In [4]:
# ==============================
# SAVE MODEL AS PKL
# ==============================
import pickle

save_data = {
    "tfidf": tfidf,
    "clf": clf,
    "sbert_model_name": "all-MiniLM-L6-v2",
    "ref_texts": ref_texts,
    "ref_labels": ref_labels,
    "ref_emb": ref_emb.cpu().numpy(),  # save as numpy
}

with open("mental_health_model.pkl", "wb") as f:
    pickle.dump(save_data, f)

print("✅ Model saved successfully as mental_health_model.pkl")

✅ Model saved successfully as mental_health_model.pkl


In [ ]:
!pip uninstall -y sentence-transformers

In [ ]:
!pip install sentence-transformers

In [30]:
!pip install transformers==4.37.2 torch==2.2.0 scikit-learn pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 2.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 63.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.4/755.4 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 82.1 MB/s eta 0:00:00:00:01:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 62.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 1.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 5.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 12.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━

In [8]:
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [9]:
df = pd.read_csv("/kaggle/input/datasets/harshmodi0710/metal-health/combined_dataset.csv")

print(df.head())

                                                text    label
0                            i didnt feel humiliated  sadness
1  i can go from feeling so hopeless to so damned...  sadness
2   im grabbing a minute to post i feel greedy wrong    anger
3  i am ever feeling nostalgic about the fireplac...     love
4                               i am feeling grouchy    anger


In [10]:
samples_per_class = 400

df_balanced = df.groupby('label').apply(lambda x: x.sample(samples_per_class, random_state=42))
df_balanced = df_balanced.reset_index(drop=True)

# shuffle data
df_balanced = df_balanced.sample(frac=1).reset_index(drop=True)

print(df_balanced['label'].value_counts())

label
sadness     400
surprise    400
love        400
anger       400
joy         400
fear        400
Name: count, dtype: int64


/tmp/ipykernel_55/241823567.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby('label').apply(lambda x: x.sample(samples_per_class, random_state=42))


In [11]:
texts = df_balanced['text'].astype(str).tolist()
labels = df_balanced['label'].tolist()

le = LabelEncoder()
labels = le.fit_transform(labels)

In [12]:
from sklearn.model_selection import train_test_split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

In [13]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(set(labels))
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

In [23]:
encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=128
)

In [24]:
# class MentalDataset(torch.utils.data.Dataset):
#     def __init__(self, encodings, labels):
#         self.encodings = encodings
#         self.labels = labels

#     def __getitem__(self, idx):
#         item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
#         item["labels"] = torch.tensor(self.labels[idx])
#         return item

#     def __len__(self):
#         return len(self.labels)

# dataset = MentalDataset(encodings, labels


class MentalDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MentalDataset(train_encodings, train_labels)
val_dataset = MentalDataset(val_encodings, val_labels)

In [25]:
# training_args = TrainingArguments(
#     output_dir="./results",
#     num_train_epochs=2,                 # fast
#     per_device_train_batch_size=8,     # faster if GPU
#     logging_dir="./logs",
#     save_strategy="no",
#     report_to="none"
# )

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    logging_dir="./logs",
    weight_decay=0.01,
    save_strategy="no",
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

NameError: name 'dataset' is not defined

In [27]:
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()

    return le.inverse_transform([predicted_class])[0]

In [28]:
def mental_health_status(label, text):
    text = text.lower()

    negative_patterns = [
        "not", "no", "can't", "dont", "never",
        "nothing makes", "no motivation", "tired of",
        "giving up", "worthless", "pressure", "stressed"
    ]

    neutral_patterns = [
        "normal day", "went to", "doing my work",
        "just another day", "nothing special", "daily routine"
    ]

    # strong negative override
    if any(p in text for p in negative_patterns):
        return "⚠️ At Risk"

    # neutral correction
    if any(p in text for p in neutral_patterns):
        return "😐 Normal"

    # model output
    if label in ["sadness", "fear", "anger"]:
        return "⚠️ At Risk"
    elif label == "neutral":
        return "😐 Normal"
    else:
        return "😊 Positive"

In [31]:
# ==============================
# IMPORTS
# ==============================
import torch
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# ==============================
# BALANCE DATASET (SAFE)
# ==============================
samples_per_class = 400

df_balanced = df.groupby('label').apply(
    lambda x: x.sample(min(len(x), samples_per_class), random_state=42)
).reset_index(drop=True)

# shuffle
df_balanced = df_balanced.sample(frac=1).reset_index(drop=True)

print(df_balanced['label'].value_counts())

# ==============================
# PREPARE DATA
# ==============================
texts = df_balanced['text'].astype(str).tolist()
labels = df_balanced['label'].tolist()

le = LabelEncoder()
labels = le.fit_transform(labels)

# ==============================
# SPLIT
# ==============================
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

# ==============================
# LOAD MODEL
# ==============================
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(set(labels))
)

# ==============================
# TOKENIZATION
# ==============================
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=128)

# ==============================
# DATASET CLASS
# ==============================
class MentalDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MentalDataset(train_encodings, train_labels)
val_dataset = MentalDataset(val_encodings, val_labels)

# ==============================
# TRAINING SETUP
# ==============================
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    weight_decay=0.01,
    save_strategy="no",
    report_to="none"
)

# ==============================
# TRAINER
# ==============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,   # ✅ FIXED
    eval_dataset=val_dataset       # ✅ ADDED
)

# ==============================
# TRAIN
# ==============================
trainer.train()

# ==============================
# EVALUATE
# ==============================
results = trainer.evaluate()
print("Validation:", results)

# ==============================
# PREDICTION FUNCTION
# ==============================
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()

    return le.inverse_transform([predicted_class])[0]

# ==============================
# FINAL MENTAL HEALTH LOGIC
# ==============================
def mental_health_status(label, text):
    text = text.lower()

    negative_patterns = [
        "not", "no", "can't", "dont", "never",
        "nothing makes", "no motivation", "tired of",
        "giving up", "worthless", "pressure", "stressed"
    ]

    neutral_patterns = [
        "normal day", "went to", "doing my work",
        "just another day", "nothing special", "daily routine"
    ]

    if any(p in text for p in negative_patterns):
        return "⚠️ At Risk"

    if any(p in text for p in neutral_patterns):
        return "😐 Normal"

    if label in ["sadness", "fear", "anger"]:
        return "⚠️ At Risk"
    elif label == "neutral":
        return "😐 Normal"
    else:
        return "😊 Positive"

# ==============================
# TEST
# ==============================
tests = [
    "I feel empty inside",
    "I am very happy today",
    "I feel nervous all the time",
    "I smile but cry inside",
    "I went to college and came back"
]

for t in tests:
    emotion = predict(t)
    status = mental_health_status(emotion, t)

    print(f"Text: {t}")
    print(f"Emotion: {emotion}")
    print(f"Mental Health: {status}")
    print("-" * 50)

/tmp/ipykernel_55/3350548425.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby('label').apply(


label
joy         400
fear        400
anger       400
sadness     400
love        400
surprise    400
Name: count, dtype: int64


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but n

Step,Training Loss


Validation: {'eval_loss': 0.4338045120239258, 'eval_runtime': 19.8111, 'eval_samples_per_second': 24.229, 'eval_steps_per_second': 3.029, 'epoch': 2.0}
Text: I feel empty inside
Emotion: sadness
Mental Health: ⚠️ At Risk
--------------------------------------------------
Text: I am very happy today
Emotion: joy
Mental Health: 😊 Positive
--------------------------------------------------
Text: I feel nervous all the time
Emotion: fear
Mental Health: ⚠️ At Risk
--------------------------------------------------
Text: I smile but cry inside
Emotion: joy
Mental Health: 😊 Positive
--------------------------------------------------
Text: I went to college and came back
Emotion: joy
Mental Health: 😐 Normal
--------------------------------------------------


In [29]:
text = "I feel very stressed and tired"

emotion = predict(text)
status = mental_health_status(emotion,text)

print("Text:", text)
print("Emotion:", emotion)
print("Mental Health:", status)

Text: I feel very stressed and tired
Emotion: surprise
Mental Health: ⚠️ At Risk


In [ ]:
trainer.train()

In [ ]:
while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    emotion = predict(user_input)
    status = mental_health_status(emotion)

    print("Emotion:", emotion)
    print("Mental Health:", status)

In [ ]:
tests_sad = [
    "I feel empty inside",
    "Nothing makes me happy anymore",
    "I am tired of everything",
    "I feel like giving up",
    "I feel useless and worthless",
    "I don’t want to talk to anyone",
    "Life feels meaningless",
    "I feel so alone even with people around",
    "I have no motivation to do anything",
    "I feel broken"
]

In [ ]:
tests_anger = [
    "I am so frustrated with everything",
    "I feel angry all the time",
    "Everything annoys me",
    "I can't control my anger",
    "I feel like shouting at everyone",
    "I hate how things are going",
    "I am under too much pressure",
    "This stress is killing me",
    "I feel overwhelmed",
    "I am losing control"
]

In [ ]:
tests_anxiety = [
    "I feel very anxious about my future",
    "I can't stop worrying",
    "I feel nervous all the time",
    "I am scared of everything",
    "My mind won't stop thinking",
    "I feel panic for no reason",
    "I can't sleep because of anxiety",
    "I overthink everything",
    "I feel uneasy constantly",
    "I am afraid of failing"
]

In [ ]:
tests_neutral = [
    "Today was a normal day",
    "I went to college and came back",
    "I am just doing my work",
    "Nothing special happened today",
    "I feel okay",
    "I am just relaxing",
    "I am doing my daily routine",
    "Everything is fine",
    "Just another day",
    "I am managing things"
]

In [ ]:
tests_happy = [
    "I feel really happy today",
    "Life is going great",
    "I am enjoying my day",
    "I feel motivated and energetic",
    "I love my life",
    "I feel confident",
    "Everything is going well",
    "I am grateful for what I have",
    "I feel peaceful",
    "I am excited about my future"
]

In [ ]:
tests_real = [
    "I smile outside but feel empty inside",
    "I am tired but trying to stay strong",
    "Sometimes I feel okay, sometimes I don’t",
    "I am stressed but handling it",
    "I feel lost in life",
    "I don’t know what I am doing anymore",
    "I feel pressure from everyone",
    "I want to be happy but I can't",
    "I feel disconnected from everything",
    "I just want peace"
]

In [ ]:
for text in all_tests:
    emotion = predict(text)
    status = mental_health_status(emotion, text)

    print(f"Text: {text}")
    print(f"Mental Health: {status}")
    print("-" * 50)

In [ ]:
tests_hidden = [
    "I smile all day but cry at night",
    "I laugh with friends but feel empty inside",
    "I look fine but I am not okay",
    "Nobody understands what I am going through",
    "I pretend to be happy but I am not",
    "I feel invisible to everyone",
    "I am tired of pretending",
    "I don't feel like myself anymore",
    "I feel numb and disconnected",
    "I am just existing, not living"
]
tests_mixed = [
    "I am happy but also stressed",
    "I feel okay but something is missing",
    "I try to stay positive but I fail",
    "I am doing well but not really",
    "I am fine but I feel lost",
    "Life is good but I feel empty",
    "I smile but I feel pain inside",
    "I am okay but sometimes I break down",
    "I am strong but I feel weak",
    "I am managing but struggling inside"
]
tests_normal = [
    "I woke up, had breakfast, and went to work",
    "I completed my assignments today",
    "I talked with my friends",
    "I watched a movie and relaxed",
    "I went for a walk in the evening",
    "I studied for my exams",
    "I cooked food and cleaned my room",
    "I attended my classes",
    "I did some exercise today",
    "I spent time with family"
]
tests_anxiety_advanced = [
    "My heart is racing and I feel scared",
    "I feel like something bad will happen",
    "I cannot control my thoughts",
    "I feel panic suddenly",
    "I am always worried about everything",
    "I feel restless and uneasy",
    "I can't calm my mind",
    "I feel trapped in my thoughts",
    "I feel constant fear",
    "I am afraid of losing control"
]
tests_positive_real = [
    "I feel peaceful and calm",
    "I am proud of myself",
    "I am improving every day",
    "I feel hopeful about my future",
    "I am grateful for today",
    "I feel relaxed and happy",
    "I am enjoying my life",
    "I feel balanced and stable",
    "I am confident in myself",
    "I feel motivated"
]

In [ ]:
new_tests = tests_hidden + tests_mixed + tests_normal + tests_anxiety_advanced + tests_positive_real

for text in new_tests:
    emotion = predict(text)
    status = mental_health_status(emotion, text)

    print(f"Text: {text}")
    print(f"Mental Health: {status}")
    print("-" * 50)

**sentence transformer**

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/harshmodi0710/metal-health/combined_dataset.csv")

print(df.head())

In [ ]:
def map_label(label):
    if label in ["sadness", "fear", "anger"]:
        return "At Risk"
    elif label == "neutral":
        return "Normal"
    else:
        return "Positive"

df['mental_health'] = df['label'].apply(map_label)

In [ ]:
texts = df['text'].astype(str).tolist()
labels = df['mental_health'].tolist()

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
X = model.encode(texts)
y = labels

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
def predict(text):
    emb = model.encode([text])
    prediction = clf.predict(emb)[0]
    return prediction

In [ ]:
tests = [
    "I feel empty inside",
    "I am okay but something is missing",
    "I feel very happy today",
    "I am stressed and tired",
    "I laugh but cry at night",
    "I feel lost in life",
    "I am confident and motivated"
]

for t in tests:
    result = predict(t)

    print(f"Text: {t}")
    print(f"Mental Health: {result}")
    print("-" * 50)

User Input
↓
Sentence Transformer (meaning understanding)
↓
Embedding vector
↓
Logistic Regression (ML model)
↓
Mental Health Output

In [ ]:

real_tests = [
    # Hidden depression
    "I smile all day but cry at night",
    "I feel empty even when I am with people",
    "I look fine but I am not okay",
    "I feel like nobody understands me",

    # Mixed emotions
    "I am happy but something feels wrong",
    "I feel okay but I am not satisfied",
    "I try to stay positive but I fail",

    # Anxiety / stress
    "I feel very stressed about my future",
    "My mind never stops thinking",
    "I feel nervous all the time",

    # Normal
    "I went to college and completed my work",
    "I had lunch and watched a movie",
    "I did my daily routine",

    # Positive
    "I feel very happy today",
    "I am confident and motivated",
    "I am enjoying my life"
]

# ==============================
# RUN TESTS
# ==============================
for text in real_tests:
    result = predict(text)

    print(f"Text: {text}")
    print(f"Predicted Mental Health: {result}")
    print("-" * 60)